<a href="https://colab.research.google.com/github/HLZHarry/Alpha-Lens-TPM/blob/main/02_feature_engineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Feature Engineering


1.   Log Returns: The standard way to measure price changes in quant finance.
2.   Momentum: Is the stock trending up or down?
3.  Volatility: How "risky" has the stock been lately?


In [ ]:
import pandas as pd
import numpy as np
from google.colab import drive

In [ ]:
#1. Load the data from Phase 1
drive.mount('/content/drive')
path = "/content/drive/MyDrive/Alpha-Lens-Project"
df = pd.read_csv(f"{path}/raw_market_data.csv")
df['Date'] = pd.to_datetime(df['Date'])

Mounted at /content/drive


In [ ]:
# Feature Engineering
# We need to calculate features PER TICKER.
# We use groupby('Ticker') to ensure AAPL's data doesn't mix with MSFT's.

# 1. Log Returns (Better for statistical modeling)
df['Log_Return'] = df.groupby('Ticker')['Price'].transform(lambda x: np.log(x / x.shift(1)))

# 2. Momentum (Return over the last 21 trading days ~ 1 month)
df['Momentum_1M'] = df.groupby('Ticker')['Log_Return'].transform(lambda x: x.rolling(window=21).sum())

# 3. Volatility (Standard Deviation of returns over 21 ydays)
df['Vol_1M'] = df.groupby('Ticker')['Log_Return'].transform(lambda x: x.rolling(window=21).std())

#4. Target Variable: Next Day's Return (What the model will try to predict)
df['Target_Next_Return'] = df.groupby('Ticker')['Log_Return'].shift(-1)

# Drop rows with NaN values created by rolling windows/shifts
df_features = df.dropna()

In [ ]:
# Save for Machine Learning
df_features.to_csv(f"{path}/engineered_features.csv", index=False)
print("Phase 2 Complete: Features engineered and saved.")
df_features.head()

Phase 2 Complete: Features engineered and saved.


,Date,Ticker,Price,Log_Return,Momentum_1M,Vol_1M,Target_Next_Return
286,2021-02-02,CNR.TO,122.043640,0.026020,-0.035841,0.018252,-0.009968
287,2021-02-02,CP.TO,87.299812,0.043296,0.029244,0.023344,-0.011101
295,2021-02-02,RY.TO,87.831017,0.010848,0.019429,0.008294,-0.001136
296,2021-02-02,SHOP.TO,158.132004,0.071332,0.124513,0.032529,-0.012178
297,2021-02-02,TD.TO,59.424534,0.015684,0.037561,0.010373,0.004051


1. Why Log Returns?
In institutional research, log returns are the standard for three reasons:

Time Additivity: Multi-day returns are calculated by summing daily log returns, whereas simple returns require multiplication.

Numerical Stability: They are symmetric (a gain and loss of equal magnitude return to zero), which helps ML models converge faster.

Stationarity: Log returns normalize price data, preventing "mathematical floors" that can confuse models.

2. Why shift(-1) for the Target?
This is the "Labeling" step for supervised learning:

The Goal: Predict the return for Tomorrow (t+1) using information available Today (t).

The Method: We "pull" tomorrow's actual return into today's row so the model can associate current features with the future outcome during the training phase.

3. Why groupby('Ticker')?
Since our data is in "Long Format" (multiple stocks in one table), we use groupby as a Data Shield:

Asset Isolation: It ensures calculations for Apple don't accidentally use data from Microsoft.

Bias Prevention: It stops "Cross-Ticker Leakage," ensuring the timeline for each asset remains independent.

4. Why transform()?
transform() is the engine of our feature engineering:

Shape Preservation: Unlike a standard average, transform() ensures the output is the same length as our input.

Mapping: It allows us to calculate a "group-level" metric (like Volatility) and map it back to every specific row for that stock.